In [1]:
import os
import asyncio
from typing import Optional, Iterable

import pandas as pd
from datasets import load_dataset

from openhands.core.logger import openhands_logger as logger
from openhands.core.main import create_runtime, run_controller
from openhands.core.config.utils import get_condenser_config_arg
from openhands.core.config.condenser_config import NoOpCondenserConfig

from evaluation.benchmarks.swe_bench.run_infer_local_docker import (
    set_dataset_type,
    get_config,
    get_instruction,
    initialize_runtime,
    complete_runtime,
)

from evaluation.utils.shared import (
    make_metadata,
    get_metrics,
    codeact_user_response,
    prepare_dataset,
)

from openhands.core.config import (
    get_llm_config_arg,
    AgentConfig,
)

IN agent_skills/repo_ops/__init__.py, imported: ['get_entity_contents', 'search_code_snippets', 'explore_tree_structure', 'get_code_structure_overview_with_code_comments']
In agent_skills.py, ['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'explore_tree_structure', 'get_code_structure_overview_with_code_comments', 'get_entity_contents', 'import_functions', 'repo_ops', 'search_code_snippets']


21:48:13 - openhands:INFO: run_infer_local_docker.py:189 - Default docker image prefix: docker.io/xingyaoww/


# ENV + HIGH-LEVEL CONFIG

In [2]:
# Path to directory with your SWE-bench docker .tar images
os.environ["LOCAL_DOCKER_IMAGE_DIR"] = (
    "/workspaces/Openhands/swebench_dockers_for_eval/"
    "swebench_verified_dockers/test/docker_images"
)

# LLM / eval config
LLM_CONFIG_NAME = "llm.together_qwen_480b"     # must exist in config.toml
CONFIG_FILE = "config.toml"

DATASET_NAME, SPLIT = "princeton-nlp/SWE-bench_Verified", "test"
AGENT_CLS = "CodeActAgent"
MAX_ITERATIONS = 50
EVAL_OUTPUT_DIR = "qwen480b_single_file_inference_test"
EVAL_NOTE = None
MODE = "swe"  # 'swe', 'swt', or 'swt-ci'

# Instance filter
SELECTED_IDS = [
    "django__django-14792",
    "astropy__astropy-14365",
    "mwaskom__seaborn-3187",
    "matplotlib__matplotlib-20676",
    "psf__requests-2931",
    "sphinx-doc__sphinx-7454",
    "pylint-dev__pylint-6386",
    "pydata__xarray-3993",
    "scikit-learn__scikit-learn-25747",
    "sympy__sympy-15976",
]
eval_n_limit = 1  # only run one instance for now

# Helpers

In [3]:
def cleanup_extra_trajectories(eval_folder):
    llm_completion_folder = os.path.join(eval_folder, "llm_completions")
    instance_ids = [d for d in os.listdir(llm_completion_folder) if os.path.isdir(os.path.join(llm_completion_folder, d))]
    for instance_id in instance_ids:

        instance_folder = os.path.join(llm_completion_folder, instance_id)

        intermediate_files = sorted(
            [f for f in os.listdir(instance_folder) if os.path.isfile(os.path.join(instance_folder, f)) and "trajectory" not in f]
        )

        # Skip the final messages json file, delete only the previous ones
        for file in intermediate_files[:-1]:
            os.remove(os.path.join(instance_folder, file))

def filter_dataset(
    dataset: pd.DataFrame,
    filter_column: str,
    selected_ids: Optional[Iterable] = None,
) -> pd.DataFrame:
    """
    Filter `dataset` to rows where `filter_column` is in `selected_ids`.

    If `selected_ids` is None, return the original dataset unchanged.
    """
    if selected_ids is None:
        print("No selected_ids provided; returning full dataset")
        return dataset

    subset = dataset[dataset[filter_column].isin(selected_ids)]
    print(
        f"Filtered dataset on {filter_column} using {len(selected_ids)} selected_ids; "
        f"retained {subset.shape[0]} / {dataset.shape[0]} rows"
    )
    return subset

# Metadata

In [4]:
llm_config = get_llm_config_arg(LLM_CONFIG_NAME, CONFIG_FILE)
llm_config.log_completions = True
llm_config.modify_params = False  # for reproducibility

condenser_name = os.environ.get("EVAL_CONDENSER")
if condenser_name:
    condenser_config = get_condenser_config_arg(condenser_name, CONFIG_FILE)
    if condenser_config is None:
        raise ValueError(
            f"Could not find Condenser config: EVAL_CONDENSER={condenser_name}"
        )
else:
    condenser_config = NoOpCondenserConfig()
    print("No Condenser config provided via EVAL_CONDENSER, using NoOpCondenser.")

dataset_description = DATASET_NAME.replace("/", "__") + "-" + SPLIT.replace("/", "__")
details = {"mode": MODE}


metadata = make_metadata(
    llm_config,
    dataset_description,
    AGENT_CLS,
    MAX_ITERATIONS,
    EVAL_NOTE,
    EVAL_OUTPUT_DIR,
    details=details,
    agent_config=None,
    condenser_config=condenser_config,
)

# Use your custom prompt template
metadata.instruction_template_name = "swe_michael.j2"
assert llm_config.native_tool_calling is True

21:48:13 - openhands:INFO: shared.py:192 - Using evaluation output directory: qwen480b_single_file_inference_test
21:48:13 - openhands:INFO: shared.py:213 - Metadata: {"agent_class":"CodeActAgent","llm_config":{"model":"together_ai/Qwen/Qwen3-Coder-480B-A35B-Instruct-FP8","api_key":"**********","base_url":"https://api.together.xyz/v1/","api_version":null,"aws_access_key_id":null,"aws_secret_access_key":null,"aws_region_name":null,"openrouter_site_url":"https://docs.all-hands.dev/","openrouter_app_name":"OpenHands","num_retries":5,"retry_multiplier":8.0,"retry_min_wait":8,"retry_max_wait":64,"timeout":null,"max_message_chars":50000,"temperature":0.0,"top_p":1.0,"top_k":null,"custom_llm_provider":null,"max_input_tokens":65384,"max_output_tokens":16384,"input_cost_per_token":null,"output_cost_per_token":null,"ollama_base_url":null,"drop_params":true,"modify_params":false,"disable_vision":null,"disable_stop_word":false,"caching_prompt":true,"log_completions":true,"log_completions_folder":"

No Condenser config provided via EVAL_CONDENSER, using NoOpCondenser.


In [5]:
llm_config

LLMConfig(model='together_ai/Qwen/Qwen3-Coder-480B-A35B-Instruct-FP8', api_key=SecretStr('**********'), base_url='https://api.together.xyz/v1/', api_version=None, aws_access_key_id=None, aws_secret_access_key=None, aws_region_name=None, openrouter_site_url='https://docs.all-hands.dev/', openrouter_app_name='OpenHands', num_retries=5, retry_multiplier=8, retry_min_wait=8, retry_max_wait=64, timeout=None, max_message_chars=50000, temperature=0.0, top_p=1.0, top_k=None, custom_llm_provider=None, max_input_tokens=65384, max_output_tokens=16384, input_cost_per_token=None, output_cost_per_token=None, ollama_base_url=None, drop_params=True, modify_params=False, disable_vision=None, disable_stop_word=False, caching_prompt=True, log_completions=True, log_completions_folder='/workspaces/OpenHands/logs/completions', custom_tokenizer=None, native_tool_calling=True, reasoning_effort='high', seed=None, safety_settings=None, for_routing=False)

# Data

In [6]:
print(f"Loading dataset {DATASET_NAME} [{SPLIT}]...")
hf_ds = load_dataset(DATASET_NAME, split=SPLIT)

# sets global DATASET_TYPE, affects docker image choice, workspace name, etc.
set_dataset_type(DATASET_NAME)

swe_bench_tests = filter_dataset(hf_ds.to_pandas(), "instance_id", selected_ids=SELECTED_IDS)
print(f"Loaded {len(swe_bench_tests)} filtered tasks")

if len(swe_bench_tests) == 0:
    raise RuntimeError("No tasks left after filtering; check SELECTED_IDS.")

output_file = os.path.join(metadata.eval_output_dir, "output.jsonl")
print(f"### OUTPUT FILE: {output_file} ###")

instances = prepare_dataset(swe_bench_tests, output_file, eval_n_limit)

# PASS_TO_PASS / FAIL_TO_PASS columns formatting, same as original
if len(instances) > 0 and not isinstance(
    instances["PASS_TO_PASS"][instances["PASS_TO_PASS"].index[0]], str
):
    for col in ["PASS_TO_PASS", "FAIL_TO_PASS"]:
        instances[col] = instances[col].apply(lambda x: str(x))

print("Instances prepared:")
print(instances.head())

# Take the first instance
instance = instances.iloc[0]
print(f"\nRunning single instance: {instance['instance_id']}")

Loading dataset princeton-nlp/SWE-bench_Verified [test]...


21:48:14 - openhands:INFO: run_infer_local_docker.py:108 - Dataset type set to: SWE-bench
21:48:14 - openhands:INFO: shared.py:231 - Writing evaluation output to qwen480b_single_file_inference_test/output.jsonl
21:48:14 - openhands:INFO: shared.py:265 - Randomly sampling 1 unique instances with random seed 42.
21:48:14 - openhands:INFO: shared.py:286 - Finished instances: 0, Remaining instances: 1


Filtered dataset on instance_id using 10 selected_ids; retained 10 / 500 rows
Loaded 10 filtered tasks
### OUTPUT FILE: qwen480b_single_file_inference_test/output.jsonl ###
Instances prepared:
                repo              instance_id  \
0  sphinx-doc/sphinx  sphinx-doc__sphinx-7454   

                                base_commit  \
0  aca3f825f2e4a8817190f3c885a242a285aa0dba   

                                               patch  \
0  diff --git a/sphinx/domains/python.py b/sphinx...   

                                          test_patch  \
0  diff --git a/tests/test_domain_py.py b/tests/t...   

                                   problem_statement hints_text  \
0  Inconsistent handling of None by `autodoc_type...              

             created_at version  \
0  2020-04-09T17:08:30Z     3.0   

                                        FAIL_TO_PASS  \
0  ["tests/test_domain_py.py::test_parse_annotati...   

                                        PASS_TO_PASS  \
0  ["tests/t

# Main

In [7]:
config = get_config(instance, metadata)
agent_config = config.get_agent_config()
agent_config.enable_cmd = True


21:48:14 - openhands:INFO: run_infer_local_docker.py:318 - Using instance container image: docker.io/swebench/sweb.eval.x86_64.sphinx-doc_1776_sphinx-7454:latest. Please make sure this image exists. Submit an issue on https://github.com/All-Hands-AI/OpenHands if you run into any issues.
21:48:14 - openhands:INFO: mapping.py:26 - Resource mapping for princeton-nlp__SWE-bench_Verified-test not found.
21:48:14 - openhands:INFO: shared.py:605 - Logging LLM completions for instance sphinx-doc__sphinx-7454 to qwen480b_single_file_inference_test/llm_completions/sphinx-doc__sphinx-7454
21:48:14 - openhands:WARNING: utils.py:778 - Model routing config section [model_routing] not found in config.toml


In [8]:
for k, v in vars(agent_config).items():
    print(f"{k}: {v}")

cli_mode: False
llm_config: None
classpath: None
system_prompt_filename: system_prompt.j2
enable_browsing: False
enable_llm_editor: False
enable_editor: True
enable_jupyter: False
enable_cmd: True
enable_think: True
enable_finish: True
enable_condensation_request: False
enable_prompt_extensions: False
enable_mcp: False
disabled_microagents: []
enable_history_truncation: True
enable_som_visual_browsing: True
enable_plan_mode: True
enable_locagent_tools_in_codeact: False
enable_alternate_locagent_tools: False
enable_code_comments_tool_in_alternate_locagent_tools: False
add_locagent_tools_first: False
use_think_plan_brainstorm_tool: False
enable_str_replace_edit_think_check: False
condenser: type='noop'
model_routing: router_name='noop_router' llms_for_routing={}
extended: ExtendedConfig()


In [ ]:
runtime = create_runtime(config)
await runtime.connect()
# await initialize_runtime(runtime, instance, metadata)  # if this is sync, drop await

21:48:14 - openhands:INFO: llm_registry.py:87 - [LLM registry d9f5fb14-15fe-4bc3-a47f-67cafb1a1d6d]: Registering service for agent
21:48:14 - openhands:INFO: docker_runtime.py:134 - Using DOCKER_HOST_IP: host.docker.internal for local_runtime_url
21:48:15 - openhands:INFO: docker_runtime.py:192 - [runtime 8a344492-9d54-47-4f994d513409250] Starting runtime with image: <Image: 'ghcr.io/all-hands-ai/runtime:oh_v0.56.0_xwu63owakdr00osp_z851pemjvcr6qui5'>


✅ Found matching folder(s):
  /workspaces/Openhands/runtime_dockers/swebench_verified/sphinx-doc__sphinx-7454


21:48:15 - openhands:INFO: docker_runtime.py:624 - [runtime 8a344492-9d54-47-4f994d513409250] Starting server with command: ['/openhands/micromamba/bin/micromamba', 'run', '-n', 'openhands', 'poetry', 'run', 'python', '-u', '-m', 'openhands.runtime.action_execution_server', '32806', '--working-dir', '/workspace', '--plugins', 'agent_skills', 'jupyter', '--username', 'root', '--user-id', '0', '--no-enable-browser']
21:48:15 - openhands:INFO: docker_runtime.py:196 - [runtime 8a344492-9d54-47-4f994d513409250] Container started: openhands-runtime-8a344492-9d54-47-4f994d513409250. VSCode URL: None
21:48:15 - openhands:INFO: docker_runtime.py:207 - [runtime 8a344492-9d54-47-4f994d513409250] Waiting for client to become ready at http://host.docker.internal:32806...


In [ ]:

# -------------------------
# 2.4 Initialize runtime (repo, env, etc.)
# -------------------------
initialize_runtime(runtime, instance, metadata)

# -------------------------
# 2.5 Build initial user message (prompt)
# -------------------------
message_action = get_instruction(instance, metadata)
print("\n===== INITIAL PROMPT =====\n")
print(message_action.content)

# -------------------------
# 2.6 Run controller (CodeActAgent lives here)
# -------------------------
state = await run_controller(
        config=config,
        initial_user_action=message_action,
        runtime=runtime,
        fake_user_response_fn=codeact_user_response,  # CodeActAgent only
    )


if state is None:
    raise RuntimeError("State is None after run_controller")

# -------------------------
# 2.7 Extract git patch
# -------------------------
return_val = complete_runtime(runtime, instance)
git_patch = return_val["git_patch"]

print("\n===== FINAL GIT PATCH =====\n")
print(git_patch)

# -------------------------
# 2.8 Metrics (optional)
# -------------------------
metrics = get_metrics(state)
print("\n===== METRICS =====\n")
print(metrics)

cleanup_extra_trajectories(metadata.eval_output_dir)


20:59:44 - openhands:INFO: run_infer_local_docker.py:397 - ------------------------------
20:59:44 - openhands:INFO: run_infer_local_docker.py:398 - BEGIN Runtime Initialization Fn
20:59:44 - openhands:INFO: run_infer_local_docker.py:399 - ------------------------------
20:59:44 - ACTION
**CmdRunAction (source=None, is_input=False)**
COMMAND:
echo 'export SWE_INSTANCE_ID=sphinx-doc__sphinx-7454' >> ~/.bashrc && echo 'export PIP_CACHE_DIR=~/.cache/pip' >> ~/.bashrc && echo "alias git='git --no-pager'" >> ~/.bashrc && git config --global core.pager "" && git config --global diff.binary false
20:59:45 - OBSERVATION
**CmdOutputObservation (source=None, exit code=0, metadata={
  "exit_code": 0,
  "pid": -1,
  "username": "root",
  "hostname": "b3a5fc2a8de9",
  "working_dir": "/workspace",
  "py_interpreter_path": "/openhands/micromamba/envs/openhands/bin/python",
  "prefix": "",
  "suffix": "\n[The command completed with exit code 0.]"
})**
--BEGIN AGENT OBSERVATION--

[The command complete


===== INITIAL PROMPT =====

<uploaded_files>
/workspace/sphinx-doc__sphinx__3.0
</uploaded_files>

I've uploaded a python code repository in the directory sphinx-doc__sphinx__3.0. Consider the following issue description:

<issue_description>
Inconsistent handling of None by `autodoc_typehints`
**Describe the bug**
With `autodoc_typehints='description'`, a function that returns `None` generates a clickable link to [None's documentation](https://docs.python.org/3/library/constants.html#None).

With `autodoc_typehints='signature'`, the `None` in the signature is not clickable.

**To Reproduce**
Steps to reproduce the behavior:
```sh
mkdir -p sphinx_type_hint_links
cd sphinx_type_hint_links

cat <<'EOF' >type_hint_test.py
def f1() -> None: return None
def f2() -> int: return 42
EOF

mkdir -p docs

cat <<'EOF' >docs/conf.py
extensions = ["sphinx.ext.autodoc", "sphinx.ext.intersphinx"]
intersphinx_mapping = {"python": ("https://docs.python.org/3", None)}
#autodoc_typehints = 'description'


21:02:46 - openhands:INFO: run_infer_local_docker.py:629 - ------------------------------
21:02:46 - openhands:INFO: run_infer_local_docker.py:630 - BEGIN Runtime Completion Fn
21:02:46 - openhands:INFO: run_infer_local_docker.py:631 - ------------------------------
21:02:46 - ACTION
**CmdRunAction (source=None, is_input=False)**
COMMAND:
cd /workspace/sphinx-doc__sphinx__3.0
21:02:47 - OBSERVATION
**CmdOutputObservation (source=None, exit code=0, metadata={
  "exit_code": 0,
  "pid": -1,
  "username": "root",
  "hostname": "b3a5fc2a8de9",
  "working_dir": "/workspace/sphinx-doc__sphinx__3.0",
  "py_interpreter_path": "/opt/miniconda3/envs/testbed/bin/python",
  "prefix": "",
  "suffix": "\n[The command completed with exit code 0.]"
})**
--BEGIN AGENT OBSERVATION--

[The command completed with exit code 0.]
[Current working directory: /workspace/sphinx-doc__sphinx__3.0]
[Python interpreter: /opt/miniconda3/envs/testbed/bin/python]
[Command finished with exit code 0]
--END AGENT OBSERVA


===== FINAL GIT PATCH =====

diff --git a/setup.py b/setup.py
index a427d5493..978e3d8e0 100644
--- a/setup.py
+++ b/setup.py
@@ -15,22 +15,22 @@ if sys.version_info < (3, 5):
     sys.exit(1)
 
 install_requires = [
-    'sphinxcontrib-applehelp',
-    'sphinxcontrib-devhelp',
+    'sphinxcontrib-applehelp<=1.0.7',
+    'sphinxcontrib-devhelp<=1.0.5',
     'sphinxcontrib-jsmath',
-    'sphinxcontrib-htmlhelp',
-    'sphinxcontrib-serializinghtml',
-    'sphinxcontrib-qthelp',
-    'Jinja2>=2.3',
+    'sphinxcontrib-htmlhelp<=2.0.4',
+    'sphinxcontrib-serializinghtml<=1.1.9',
+    'sphinxcontrib-qthelp<=1.0.6',
+    'Jinja2<3.0',
     'Pygments>=2.0',
     'docutils>=0.12',
     'snowballstemmer>=1.1',
     'babel>=1.3',
-    'alabaster>=0.7,<0.8',
+    'alabaster>=0.7,<0.7.12',
     'imagesize',
     'requests>=2.5.0',
     'setuptools',
-    'packaging',
+    'packaging', 'markupsafe<=2.0.1',
 ]
 
 extras_require = {
diff --git a/tox.ini b/tox.ini
index d9f040544..bf39854b6 100644

In [ ]:
dir(state)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_history_checksum',
 '_view',
 'agent_state',
 'budget_flag',
 'confirmation_mode',
 'conversation_stats',
 'delegate_level',
 'delegates',
 'end_id',
 'extra_data',
 'get_current_user_intent',
 'get_last_agent_message',
 'get_last_user_message',
 'get_local_metrics',
 'get_local_step',
 'history',
 'inputs',
 'iteration',
 'iteration_flag',
 'last_error',
 'local_iteration',
 'local_metrics',
 'max_iterations',
 'metrics',
 'outputs',
 'parent_iteration',
 'parent_metrics_snapshot',
 'restore_from_s

In [ ]:
state.history

[SystemMessageAction(content='You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks.\n\n<ROLE>\nYour primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed.\n* If the user asks a question, like "why is X happening", don\'t try to fix the problem. Just give an answer to the question.\n</ROLE>\n\n<EFFICIENCY>\n* Each action you take is somewhat expensive. Wherever possible, combine multiple actions into a single action, e.g. combine multiple bash commands into one, using sed and grep to edit/view multiple files at once.\n* When exploring the codebase, use efficient tools like find, grep, and git commands with appropriate filters to minimize unnecessary operations.\n</EFFICIENCY>\n\n<FILE_SYSTEM_GUIDELINES>\n* When a user provides a file path, do NOT assume it\'s relative to the current working directory. First explore

In [ ]:
dir(config)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__',
 '__pydantic_setattr_handl